# Handwritten Digit Recognition using K-Nearest Neighbors (KNN)

Full MNIST dataset (70,000 samples) from Yann LeCun's original source.

**Steps:**
1. Data Loading & Exploration
2. Data Preprocessing & Visualization
3. KNN from Scratch
4. KNN using Scikit-Learn
5. Model Evaluation (Accuracy, Confusion Matrix, Classification Report)
6. Hyperparameter Tuning (Finding Optimal K)
7. Save & Load Model
8. Predict on Custom Samples

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Data Loading & Exploration

In [ ]:
# Load the MNIST dataset
df = pd.read_csv('data/train.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Number of samples: {df.shape[0]}')
print(f'Number of features (pixels + label): {df.shape[1]}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Check for missing values
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nData types:\n{df.dtypes.value_counts()}')
print(f'\nBasic Statistics:')
df.describe()

In [ ]:
# Class distribution
plt.figure(figsize=(10, 5))
digit_counts = df['label'].value_counts().sort_index()
bars = plt.bar(digit_counts.index, digit_counts.values, color=plt.cm.tab10(np.arange(10)/10))
plt.xlabel('Digit', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Digits in Dataset', fontsize=14)
plt.xticks(range(10))
for bar, count in zip(bars, digit_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/digit_distribution.png', dpi=150)
plt.show()

## 3. Data Preprocessing & Visualization

In [ ]:
# Separate features and labels
X = df.drop('label', axis=1).values
y = df['label'].values

print(f'Features shape: {X.shape}')
print(f'Labels shape: {y.shape}')
print(f'Pixel value range: [{X.min()}, {X.max()}]')

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Sample Handwritten Digits from MNIST', fontsize=14, y=1.02)

for digit in range(10):
    # Get two samples of each digit
    indices = np.where(y == digit)[0]
    for row in range(2):
        ax = axes[row, digit]
        img = X[indices[row]].reshape(28, 28)
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/sample_digits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Normalize pixel values to [0, 1]
X_normalized = X / 255.0

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

## 4. KNN Implementation from Scratch

In [ ]:
class KNNFromScratch:
    """K-Nearest Neighbors classifier implemented from scratch."""
    
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
    
    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict_single(self, query_point):
        # Compute distances to all training points
        distances = []
        for i in range(self.X_train.shape[0]):
            d = self.euclidean_distance(self.X_train[i], query_point)
            distances.append((d, self.y_train[i]))
        
        # Sort by distance and pick K nearest
        distances.sort(key=lambda x: x[0])
        k_nearest = distances[:self.k]
        
        # Majority vote
        labels = [label for _, label in k_nearest]
        most_common = Counter(labels).most_common(1)
        return most_common[0][0]
    
    def predict(self, X_test):
        predictions = []
        for i in range(X_test.shape[0]):
            pred = self.predict_single(X_test[i])
            predictions.append(pred)
        return np.array(predictions)

In [ ]:
# Test the scratch implementation on a small subset (full dataset is slow without optimization)
print('Testing KNN from scratch on 200 test samples...')
print('(Using full training set for lookup)\n')

# Use a subset of training data for the scratch version (for speed)
scratch_train_size = 5000
scratch_test_size = 200

knn_scratch = KNNFromScratch(k=5)
knn_scratch.fit(X_train[:scratch_train_size], y_train[:scratch_train_size])

start_time = time.time()
y_pred_scratch = knn_scratch.predict(X_test[:scratch_test_size])
scratch_time = time.time() - start_time

scratch_accuracy = accuracy_score(y_test[:scratch_test_size], y_pred_scratch)
print(f'KNN from Scratch Results:')
print(f'  Accuracy: {scratch_accuracy * 100:.2f}%')
print(f'  Time taken: {scratch_time:.2f} seconds')
print(f'  Training samples used: {scratch_train_size}')
print(f'  Test samples evaluated: {scratch_test_size}')

## 5. KNN using Scikit-Learn (Optimized)

In [ ]:
# Train KNN using scikit-learn (much faster with optimized data structures)
print('Training scikit-learn KNN on full dataset...\n')

knn_sklearn = KNeighborsClassifier(n_neighbors=4, weights='distance', algorithm='auto', n_jobs=-1)

start_time = time.time()
knn_sklearn.fit(X_train, y_train)
train_time = time.time() - start_time
print(f'Training time: {train_time:.2f} seconds')

start_time = time.time()
y_pred_sklearn = knn_sklearn.predict(X_test)
predict_time = time.time() - start_time
print(f'Prediction time: {predict_time:.2f} seconds')

sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)
print(f'\nScikit-Learn KNN Accuracy: {sklearn_accuracy * 100:.2f}%')

## 6. Model Evaluation

In [ ]:
# Classification Report
print('Classification Report:')
print('=' * 60)
print(classification_report(y_test, y_pred_sklearn, digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_sklearn)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - KNN Digit Recognition', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150)
plt.show()

# Per-digit accuracy
print('\nPer-Digit Accuracy:')
for digit in range(10):
    mask = y_test == digit
    digit_acc = accuracy_score(y_test[mask], y_pred_sklearn[mask])
    print(f'  Digit {digit}: {digit_acc * 100:.2f}%')

In [ ]:
# Visualize some predictions
fig, axes = plt.subplots(3, 10, figsize=(16, 6))
fig.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14, y=1.02)

sample_indices = np.random.choice(len(X_test), 30, replace=False)

for idx, ax in enumerate(axes.flat):
    i = sample_indices[idx]
    img = X_test[i].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    
    pred = y_pred_sklearn[i]
    true = y_test[i]
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P:{pred} T:{true}', fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show misclassified examples
misclassified = np.where(y_pred_sklearn != y_test)[0]
print(f'Total misclassified: {len(misclassified)} out of {len(y_test)} ({len(misclassified)/len(y_test)*100:.2f}%)\n')

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Misclassified Digits', fontsize=14, y=1.02)

for idx, ax in enumerate(axes.flat):
    if idx < len(misclassified):
        i = misclassified[idx]
        img = X_test[i].reshape(28, 28)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'P:{y_pred_sklearn[i]} T:{y_test[i]}', fontsize=9, color='red', fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('outputs/misclassified.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hyperparameter Tuning - Finding Optimal K

In [ ]:
# Find the best value of K (with distance weighting)
k_values = range(1, 16)
accuracies = []

print('Finding optimal K value (distance-weighted)...')
for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance', algorithm='auto', n_jobs=-1)
    knn_temp.fit(X_train, y_train)
    y_pred_temp = knn_temp.predict(X_test)
    acc = accuracy_score(y_test, y_pred_temp)
    accuracies.append(acc)
    print(f'  K={k:2d} -> Accuracy: {acc * 100:.2f}%')

best_k = k_values[np.argmax(accuracies)]
best_acc = max(accuracies)
print(f'\nBest K = {best_k} with accuracy = {best_acc * 100:.2f}%')

In [ ]:
# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_values, [a * 100 for a in accuracies], 'bo-', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='r', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('KNN: Accuracy vs K Value', fontsize=14)
plt.xticks(k_values)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/k_vs_accuracy.png', dpi=150)
plt.show()

## 8. Train Final Model with Best K & Save

In [ ]:
# Train final model with best K
print(f'Training final model with K={best_k} (distance-weighted)...')
final_model = KNeighborsClassifier(n_neighbors=best_k, weights='distance', algorithm='auto', n_jobs=-1)
final_model.fit(X_train, y_train)

final_accuracy = accuracy_score(y_test, final_model.predict(X_test))
print(f'Final Model Accuracy: {final_accuracy * 100:.2f}%')

# Save the model
with open('models/knn_digit_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print('\nModel saved to models/knn_digit_model.pkl')

## 9. Predict on Custom Samples

In [ ]:
# Load saved model and predict
with open('models/knn_digit_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Pick random test samples and predict
random_indices = np.random.choice(len(X_test), 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Predictions from Saved Model', fontsize=14)

for idx, ax in enumerate(axes.flat):
    i = random_indices[idx]
    img = X_test[i].reshape(28, 28)
    prediction = loaded_model.predict(X_test[i].reshape(1, -1))[0]
    
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    color = 'green' if prediction == y_test[i] else 'red'
    ax.set_title(f'Predicted: {prediction}', fontsize=11, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/final_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nProject Complete!')
print(f'Final Model: KNN with K={best_k}')
print(f'Accuracy: {final_accuracy * 100:.2f}%')